<a href="https://colab.research.google.com/github/hy961/Case-Studies-for-Data-Science/blob/main/Case_Studies_for_Data_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# Dataset 2 HateXplain
# Models used SVM (LinearSVC) and  Logistic Regression (as baseline)

import json
import pandas as pd
from collections import Counter
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from google.colab import files

RANDOM_STATE = 999

# Upload the dataset
from google.colab import files
uploaded = files.upload()

# Import the dataset
with open("dataset.json", "r") as f:
    raw = json.load(f)

# Preprocessing
#    Each post has 3 annotators. Take the majority label/ Drop posts where all 3 disagree (no clear majority)
#    Then, reconstruct text from the token list.
# The 'raw' variable is already loaded from the file above, no need to open it again.

rows = []
dropped = 0
for post_id, entry in raw.items():
    labels = [ann["label"] for ann in entry["annotators"]]
    majority_label, count = Counter(labels).most_common(1)[0]
    if count >= 2:
        rows.append({
            "post_id": post_id,
            "text": " ".join(entry["post_tokens"]),
            "label": majority_label
        })
    else:
        dropped += 1

hx = pd.DataFrame(rows)

print("Dataset Summary")
print("=" * 50)
print(f"Total posts in raw file : {len(raw)}")
print(f"Dropped (no majority)   : {dropped}")
print(f"Retained for modelling  : {len(hx)}")
print()
print("Class distribution:")
print(hx["label"].value_counts())
print()
print("Sample rows:")
print(hx.head())
print()

# Save cleaned version (for appendix)
hx.to_csv("hatexplain_clean.csv", index=False)


# Train/test split & TF-IDF
X_train, X_test, y_train, y_test = train_test_split(
    hx["text"], hx["label"],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=hx["label"]
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape :", X_test_tfidf.shape)
print()

# Model 1: SVM ( it is a new algorithm, not used in previous  ML course)
svm = LinearSVC(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    max_iter=5000
)
svm.fit(X_train_tfidf, y_train)
svm_preds = svm.predict(X_test_tfidf)

print("=" * 50)
print("Model 1: SVM (LinearSVC)")
print("=" * 50)
print(classification_report(y_test, svm_preds, digits=3))
print("Macro F1:", round(f1_score(y_test, svm_preds, average="macro"), 3))
print()
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, svm_preds, labels=sorted(hx["label"].unique())),
    index=sorted(hx["label"].unique()),
    columns=sorted(hx["label"].unique())
))
print()

feats = np.array(vectorizer.get_feature_names_out())

print("=" * 50)
print("Top weighted features per class (SVM)")
print("=" * 50)
for i, cls in enumerate(svm.classes_):
    top = np.argsort(svm.coef_[i])[::-1][:12]
    print(f"\n{cls}:")
    print("  " + ", ".join(feats[top]))
print()

# Model 2: Logistic Regression
logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
logreg.fit(X_train_tfidf, y_train)
logreg_preds = logreg.predict(X_test_tfidf)

print("=" * 50)
print("Model 2: Logistic Regression")
print("=" * 50)
print(classification_report(y_test, logreg_preds, digits=3))
print("Macro F1:", round(f1_score(y_test, logreg_preds, average="macro"), 3))
print()
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, logreg_preds, labels=sorted(hx["label"].unique())),
    index=sorted(hx["label"].unique()),
    columns=sorted(hx["label"].unique())
))
print()


# Summaries
summary = pd.DataFrame({
    "Model": ["SVM (LinearSVC)", "Logistic Regression"],
    "Accuracy": [
        round((svm_preds == y_test).mean(), 3),
        round((logreg_preds == y_test).mean(), 3)
    ],
    "Macro F1": [
        round(f1_score(y_test, svm_preds, average="macro"), 3),
        round(f1_score(y_test, logreg_preds, average="macro"), 3)
    ],
    "Weighted F1": [
        round(f1_score(y_test, svm_preds, average="weighted"), 3),
        round(f1_score(y_test, logreg_preds, average="weighted"), 3)
    ]
})

print("=" * 50)
print("Summary of Dataset 2")
print("=" * 50)
print(summary.to_string(index=False))

Saving dataset.json to dataset (1).json
Dataset Summary
Total posts in raw file : 20148
Dropped (no majority)   : 919
Retained for modelling  : 19229

Class distribution:
label
normal        7814
hatespeech    5935
offensive     5480
Name: count, dtype: int64

Sample rows:
                       post_id  \
0  1179055004553900032_twitter   
1  1179063826874032128_twitter   
2  1178793830532956161_twitter   
3  1179088797964763136_twitter   
4  1179085312976445440_twitter   

                                                text       label  
0  i dont think im getting my baby them white 9 h...      normal  
1  we cannot continue calling ourselves feminists...      normal  
2                      nawt yall niggers ignoring me      normal  
3  <user> i am bit confused coz chinese ppl can n...  hatespeech  
4  this bitch in whataburger eating a burger with...  hatespeech  

Train shape: (15383, 10000)
Test shape : (3846, 10000)

Model 1: SVM (LinearSVC)
              precision    recall  f1

In [13]:
# Dataset 1 Jigsaw Toxic Comment Classification
# Models used SVM (LinearSVC) and Logistic Regression, One-vs-Rest

import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

RANDOM_STATE = 999

from google.colab import files
uploaded = files.upload()

jig = pd.read_csv("train.csv")
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Dataset Summary")
print("=" * 50)
print("Shape:", jig.shape)
print("Columns:", list(jig.columns))
print("\nPositive count per label:")
print(jig[label_cols].sum())

n_clean = (jig[label_cols].sum(axis=1) == 0).sum()
print(f"\nComments with NO label: {n_clean} ({n_clean / len(jig):.1%})")
print("\nLabels per comment (how many carry multiple):")
print(jig[label_cols].sum(axis=1).value_counts().sort_index())

X_train, X_test, y_train, y_test = train_test_split(
    jig["comment_text"], jig[label_cols],
    test_size=0.2,
    random_state=RANDOM_STATE
)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

# using OneVsRestClassifier here as this is multi-label, a comment
# can be toxic & obscene & an insult all at the same time
results = []

def run_model(name, base_estimator):
    clf = OneVsRestClassifier(base_estimator)
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)

    print(f"\n{name} , Dataset 1")
    print(classification_report(
        y_test, preds, target_names=label_cols, digits=3, zero_division=0
    ))

    micro_f1 = f1_score(y_test, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    micro_p = precision_score(y_test, preds, average="micro", zero_division=0)
    micro_r = recall_score(y_test, preds, average="micro", zero_division=0)

    print("Micro Precision:", round(micro_p, 3))
    print("Micro Recall:", round(micro_r, 3))
    print("Micro F1:", round(micro_f1, 3))
    print("Macro F1:", round(macro_f1, 3))

    results.append({
        "Model": name,
        "Precision": round(micro_p, 3),
        "Recall": round(micro_r, 3),
        "Micro F1": round(micro_f1, 3),
        "Macro F1": round(macro_f1, 3)
    })

    return clf

# using balanced class weights here, compensates for the heavy
# imbalance and pushes recall up hard
run_model(
    "SVM (LinearSVC, balanced)",
    LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
)

run_model(
    "Logistic Regression (balanced)",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
)

# no class balancing this round. expecting precision to rise and
# recall to fall, which is the trade-off will be covered in the report
svm_default = run_model(
    "SVM (LinearSVC, default)",
    LinearSVC(max_iter=5000, random_state=RANDOM_STATE)
)

run_model(
    "Logistic Regression (default)",
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
)

feats = np.array(vectorizer.get_feature_names_out())

print("=" * 50)
print("Top weighted features per label (SVM, default weights)")
print("=" * 50)
for i, lab in enumerate(label_cols):
    coefs = svm_default.estimators_[i].coef_[0]
    top = np.argsort(coefs)[::-1][:12]
    print(f"\n{lab}:")
    print("  " + ", ".join(feats[top]))
print()

summary = pd.DataFrame(results)

print("=" * 50)
print("\nSummary of Dataset 1")
print("=" * 50)
print(summary.to_string(index=False))

Saving train.csv to train (1).csv
Dataset Summary
Shape: (159571, 8)
Columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Positive count per label:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Comments with NO label: 143346 (89.8%)

Labels per comment (how many carry multiple):
0    143346
1      6360
2      3480
3      4209
4      1760
5       385
6        31
Name: count, dtype: int64
Train shape: (127656, 10000)
Test shape: (31915, 10000)

SVM (LinearSVC, balanced) , Dataset 1
               precision    recall  f1-score   support

        toxic      0.585     0.839     0.689      3169
 severe_toxic      0.251     0.696     0.369       326
      obscene      0.599     0.852     0.704      1719
       threat      0.304     0.546     0.391       108
       insult      0.474     0.828     0.603      1612
identity_hate      0.216 